# Constructing binary phase diagrams from Gibbs free-energy curves

Copyright © 2026 Philip Eisenlohr and contributors. [License](https://github.com/mseMSU/Notebooks-pub/blob/main/LICENSE.md)

## Overview

This notebook explores how temperature changes the molar Gibbs free energy of solid and liquid solutions in a hypothetical binary A–B system.
At each selected temperature, the stable state is determined by the lowest attainable Gibbs free energy.
A common-tangent construction identifies compositions that can coexist in equilibrium.
Repeating that construction at several temperatures provides the information needed to sketch a binary phase diagram.

After working through the notebook, you should be able to:

- Explain the pure-component, enthalpic, and entropic contributions to a solution's Gibbs free energy.
- Predict qualitatively how increasing temperature changes the free-energy curves.
- Compare solid and liquid free-energy curves at several chosen temperatures.
- Use a common tangent to estimate equilibrium phase compositions.
- Transfer those compositions to an externally prepared temperature–composition phase diagram.

The parameters below are selected for instruction and do not describe a specific alloy or replace assessed thermodynamic data.

## Why Gibbs free-energy curves determine phase equilibrium

At fixed temperature and pressure, a closed system approaches the accessible state with the lowest total Gibbs free energy.
For a binary system, the horizontal coordinate of a free-energy curve is composition and the vertical coordinate is molar Gibbs free energy.
We use $x_B$ for the mole fraction of component B and $x_A=1-x_B$ for the mole fraction of component A.
The left and right limits therefore represent pure A and pure B, respectively.

A single homogeneous phase is stable at a composition when its free-energy curve forms the lower convex envelope of all available states.
A straight line tangent to two points on the available curves represents a two-phase mixture whose overall free energy lies below the homogeneous alternatives between those compositions.
The tangent points give the equilibrium compositions of the coexisting phases.
This construction is commonly called either the **common-tangent** or **double-tangent** construction.

## Regular-solution model

We consider one solid phase and one liquid phase, each of which can span the complete composition range.
The molar Gibbs free energy of phase $\phi$ is represented by

$$
G_m^{\phi}(x_B,T)
=x_A G_A^{\phi}(T)+x_B G_B^{\phi}(T)
+\Omega^{\phi}x_Ax_B
+RT\left(x_A\ln x_A+x_B\ln x_B\right).
$$

The first two terms interpolate between the Gibbs free energies of the pure components in that phase.
The regular-solution parameter $\Omega^{\phi}$ describes the non-ideal enthalpy of mixing.
The enthalpy contribution resulting from a positive $\Omega^{\phi}$ penalizes unlike neighbors and can favor separation, whereas the configurational entropy term always favors mixing.
Because the entropy contribution is multiplied by temperature, its stabilizing influence on homogeneous mixing generally grows as temperature increases.

Before running the calculations, predict which term will dominate at sufficiently high temperature and how that should affect the curvature of the free-energy curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    },
)
# Keep False for portable rendering; use True only with a complete TeX installation.
plt.rcParams["text.usetex"] = False

gas_constant = 8.314             # Ideal-gas constant, J/(mol K)
n_sampling = 101                 # Number of sampled compositions and temperatures


## Gibbs free energy of the pure components

Each solution curve must approach the appropriate pure-component free energy at $x_B=0$ and $x_B=1$.
For this exercise, those reference energies are represented by the simple polynomial

$$
G_k^{\phi}(T)=a_k^{\phi}+b_k^{\phi}T^2+c_k^{\phi}T^3,
$$

where $k$ denotes A or B and $\phi$ denotes solid or liquid.
The four parameter triplets below create distinct temperature dependences for solid A, liquid A, solid B, and liquid B.
Their values are pedagogical rather than a thermodynamic assessment.

Where the solid and liquid curves for a pure component cross, the model assigns equal Gibbs free energy to those two states.
Before revealing the plot, identify what such a crossing represents physically.

In [ ]:
def pure_gibbs_energy(T, a=0.0, b=0.0, c=0.0):
    """Return the modeled molar Gibbs free energy in J/mol."""
    return a + b * T**2 + c * T**3


A_liquid = (1200.0, -2e-4, -6e-6)
B_liquid = (900.0, -8e-3, 0.0)
A_solid = (-500.0, -1e-3, -3e-6)
B_solid = (500.0, -2e-3, 0.0)


In [ ]:
temperature_grid = np.linspace(0.0, 1000.0, n_sampling)
component_colors = sns.color_palette("colorblind", n_colors=2)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.lineplot(
    x=temperature_grid, y=pure_gibbs_energy(temperature_grid, *A_solid),
    color=component_colors[0], linestyle="-", label="Solid A", ax=ax
)
sns.lineplot(
    x=temperature_grid, y=pure_gibbs_energy(temperature_grid, *A_liquid),
    color=component_colors[0], linestyle=":", linewidth=2.5,
    label="Liquid A", ax=ax
)
sns.lineplot(
    x=temperature_grid, y=pure_gibbs_energy(temperature_grid, *B_solid),
    color=component_colors[1], linestyle="-", label="Solid B", ax=ax
)
sns.lineplot(
    x=temperature_grid, y=pure_gibbs_energy(temperature_grid, *B_liquid),
    color=component_colors[1], linestyle=":", linewidth=2.5,
    label="Liquid B", ax=ax
)
ax.set_xlabel("Temperature / K")
ax.set_ylabel(r"Molar Gibbs free energy / J mol$^{-1}$")
ax.set_title("Pure-component reference energies")
ax.legend()
fig.tight_layout()
plt.show()

### Interpreting the pure-component crossings

At a crossing, the modeled solid and liquid forms of that pure component have equal molar Gibbs free energy.
At the specified pressure, this temperature represents the model's equilibrium melting temperature for the pure component.
Away from the crossing, the phase with the lower reference energy is the stable pure state.
These endpoint preferences anchor the solid and liquid solution curves developed next.

## Gibbs free-energy curves of the solid and liquid solutions

The function below combines the pure-component references, regular-solution enthalpy, and ideal configurational entropy.
The exact endpoints are excluded from the numerical composition grid because $\ln 0$ is undefined, although the limiting entropy contribution is zero.

The solid uses a larger positive interaction parameter than the liquid in this model.
Before plotting, consider how that difference should influence the curvature of the two phases.

In [ ]:
def solution_gibbs_energy(T, composition, A_parameters, B_parameters, omega):
    """Return the regular-solution molar Gibbs free energy in J/mol."""
    pure_contribution = (
        (1.0 - composition) * pure_gibbs_energy(T, *A_parameters)
        + composition * pure_gibbs_energy(T, *B_parameters)
    )
    enthalpy_of_mixing = omega * (1.0 - composition) * composition
    entropy_contribution = gas_constant * T * (
        (1.0 - composition) * np.log(1.0 - composition)
         + composition * np.log(composition)
    )
    return pure_contribution + enthalpy_of_mixing + entropy_contribution


def solution_free_energy_curves(temperature, composition):
    """Return the solid and liquid free-energy curves at one temperature."""
    solid = solution_gibbs_energy(temperature, composition, A_solid, B_solid, omega_solid)
    liquid = solution_gibbs_energy(temperature, composition, A_liquid, B_liquid, omega_liquid)
    return solid, liquid


def plot_free_energy_curves(temperature, composition, ax=None, ylim=None, title=None):
    """Plot the absolute solid and liquid free-energy curves."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(4.0, 6.0))
    else:
        fig = ax.figure

    solid, liquid = solution_free_energy_curves(temperature, composition)
    sns.lineplot(
        x=composition, y=solid, color=phase_colors[0], linewidth=2,
        label="Solid", legend=False, ax=ax
    )
    sns.lineplot(
        x=composition, y=liquid, color=phase_colors[1], linewidth=2,
        label="Liquid", legend=False, ax=ax
    )
    ax.set_title(title or f"T = {temperature:.0f} K")
    ax.set_xlim(composition[0], composition[-1])
    if ylim is not None:
        ax.set_ylim(*ylim)
    if standalone:
        ax.set_xlabel(r"Mole fraction of B, $x_B$")
        ax.set_ylabel(r"Molar Gibbs free energy / J mol$^{-1}$")
        ax.legend()
        fig.tight_layout()

    return fig, ax


def padded_limits(*values, ticks=5):
    maximum = np.max(values)
    minimum = np.min(values)

    tick = np.power(10, np.round(np.log10((maximum - minimum) / ticks)))
    return (
        np.floor(minimum / tick) * tick,
        np.ceil(maximum / tick) * tick,
    )


In [ ]:
omega_liquid = 2000.0            # Liquid interaction parameter, J/mol
omega_solid = 18000.0            # Solid interaction parameter, J/mol

composition_grid = np.linspace(1e-6, 1.0 - 1e-6, n_sampling)
phase_colors = sns.color_palette("colorblind", n_colors=2)


## Select temperatures for the phase-equilibrium exercise

Edit the `temperatures` list in the next cell to choose the isothermal sections you want to investigate.
Using a list makes every selected curve visible in the static GitBook version and records the choices when the notebook is saved.
The initial set is ordered from 1000 K down to 300 K and begins slightly above the higher modeled pure-component melting temperature of approximately 927 K.

Each panel displays the solid and liquid curves at one temperature, and all panels share the same absolute Gibbs-energy range.
The shared scale makes temperature-dependent changes easier to compare, although phase competition must still be evaluated between curves at the same temperature.
Before revealing the plots, predict which phase will be favored near pure A and pure B at each selected temperature.

In [ ]:
temperatures = [1000, 750, 500, 300]  # Selected temperatures, K

if not temperatures:
    raise ValueError("Select at least one temperature.")

selected_curves = [
    curve
    for temperature in temperatures
    for curve in solution_free_energy_curves(temperature, composition_grid)
]
selected_energy_ylim = padded_limits(*selected_curves)
fig, axes = plt.subplots(
    1, len(temperatures), figsize=(3*len(temperatures), 6),
    sharex=True, sharey=True, squeeze=False,
)
axes = axes.ravel()

for ax, temperature in zip(axes, temperatures):
    plot_free_energy_curves(temperature, composition_grid, ax=ax, ylim=selected_energy_ylim)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncols=2)
fig.supxlabel(r"Mole fraction of B, $x_B$")
fig.supylabel(r"Molar Gibbs free energy / J mol$^{-1}$")
fig.tight_layout(rect=(0.03, 0.03, 1.0, 0.93))
plt.show()


## Explore the temperature dependence interactively

The four panels above preserve several selected temperatures for comparison and for constructing common tangents.
When this notebook is running in MSU JupyterHub, Binder, Colab, or a local JupyterLab environment, the slider below provides a continuous view between the temperature limits defined in the next cell.
The upper limit is slightly above the higher modeled pure-component melting temperature of approximately 927 K.

The interactive figure presents three synchronized views at the selected temperature: the complete composition range, near-pure A, and near-pure B.
The terminal views resolve the first and last few mole percent of B and use independently sampled composition grids so their curves remain smooth.

The full-range panel uses absolute Gibbs free energies and one fixed vertical range established from the curves at the two temperature limits.
Each terminal panel instead recalculates its own vertical range at the selected temperature so small differences near the pure compositions remain visible.


In [ ]:
interactive_temperature_min = 200.0   # Lower end of temperature range, K
interactive_temperature_max = 1000.0  # Above the higher melting point, K
interactive_energy_ylim = padded_limits(
    *solution_free_energy_curves(interactive_temperature_min, composition_grid),
    *solution_free_energy_curves(interactive_temperature_max, composition_grid),
)

terminal_fraction = 0.05
near_pure_A_composition = np.linspace(composition_grid[0], terminal_fraction, n_sampling)
near_pure_B_composition = np.linspace(1.0 - terminal_fraction, composition_grid[-1], n_sampling)
interactive_composition_views = [
    ("Full range", composition_grid, interactive_energy_ylim),
    ("Near-pure A", near_pure_A_composition, None),
    ("Near-pure B", near_pure_B_composition, None),
]


In [ ]:
try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    get_ipython().run_line_magic("pip", "install -q ipywidgets")
    import ipywidgets as widgets

from IPython.display import display


temperature_slider = widgets.FloatSlider(
    value=500.0,
    min=interactive_temperature_min,
    max=interactive_temperature_max,
    step=10.0,
    description="T / K",
    continuous_update=True,
    readout_format=".0f",
    layout=widgets.Layout(width="95%"),
)


def update_free_energy_curves(temperature):
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 6.0), sharey=False, squeeze=False)
    axes = axes.ravel()
    for ax, (title, composition, energy_ylim) in zip(axes, interactive_composition_views):
        curves = solution_free_energy_curves(temperature, composition)
        if energy_ylim is None:
            energy_ylim = padded_limits(*curves)
        plot_free_energy_curves(
            temperature, composition, ax=ax,
            ylim=energy_ylim, title=title
        )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.suptitle(f"T = {temperature:.0f} K", y=0.995)
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.94), ncols=2)
    fig.supxlabel(r"Mole fraction of B, $x_B$")
    fig.supylabel(r"Molar Gibbs free energy / J mol$^{-1}$")
    fig.tight_layout(rect=(0.03, 0.03, 1.0, 0.88))
    plt.show()
    plt.close(fig)


interactive_plot = widgets.interactive_output(
    update_free_energy_curves, {"temperature": temperature_slider}
)
display(widgets.VBox([temperature_slider, interactive_plot]))


## Construct common tangents

At a chosen temperature, suppose a tangent touches phase $\alpha$ at composition $x_B^{\alpha}$ and phase $\beta$ at $x_B^{\beta}$.
The slopes at the two contact points and the slope of the line joining them must agree:

$$
\left.\frac{\mathrm{d}G_m^{\alpha}}{\mathrm{d}x_B}\right|_{x_B^{\alpha}}
=\left.\frac{\mathrm{d}G_m^{\beta}}{\mathrm{d}x_B}\right|_{x_B^{\beta}}
=\frac{G_m^{\beta}(x_B^{\beta})-G_m^{\alpha}(x_B^{\alpha})}
{x_B^{\beta}-x_B^{\alpha}}.
$$

Thermodynamically, this equality means that the chemical potential of each component (A and B) is the same in both phases ($\alpha$ and $\beta$).
Geometrically, the candidate tangent must also lie at or below the relevant free-energy curves between its contact points.
A line that merely intersects both curves or is tangent to only one does not establish two-phase equilibrium.

For each temperature, draw the common tangent externally on a printed plot or in suitable drawing software.
Record the two contact compositions and identify the phase represented by each contact point.
Depending on temperature and model parameters, a common tangent may connect solid and liquid curves or two separated compositions on the same phase curve.

## Build the temperature–composition phase diagram

Use the following workflow to convert the isothermal free-energy information into a phase diagram:

1. Select a series of temperatures that spans the changes visible in the free-energy curves.
2. Construct the physically admissible common tangent at each temperature.
3. Record the composition and phase at every tangent point.
4. On a separate graph, place mole fraction $x_B$ on the horizontal axis and temperature on the vertical axis.
5. Plot each recorded solid composition and liquid composition at its corresponding temperature.
6. Connect related points smoothly to estimate the phase boundaries.
7. Label the single-phase and two-phase regions implied by those boundaries.

Add more temperatures where a boundary changes rapidly or where its location remains uncertain.
Your phase diagram should be treated as a numerical construction from this simplified model rather than as data for a real alloy system.

## Try next

1. Increase or decrease `omega_solid` and predict how the solid curve and phase boundaries will respond.
2. Set both interaction parameters to zero to recover ideal-solution behavior.
3. Add temperatures near an apparent invariant or terminal feature to resolve it more accurately.
4. Compare your constructed diagram with one generated by numerically solving the common-tangent equations.
5. Replace the pedagogical parameters with a documented thermodynamic model and identify which additional physical contributions are required.